In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv('/content/student_depression_dataset.csv')

df = df.drop(df[df['Profession'] != 'Student'].index)
df = df.drop(df[df['Work Pressure'] != 0.0].index)
df = df.drop(df[df['Job Satisfaction'] != 0.0].index)

col = ['id', 'Profession', 'Work Pressure', 'Job Satisfaction']
df = df.drop(columns=col)

df = df.replace(r'^\s*[\?]\s*$', np.nan, regex=True)
df['Financial Stress'] = pd.to_numeric(df['Financial Stress'], errors='coerce')

df['Sleep Duration'] = df['Sleep Duration'].astype(str).str.replace("'", "").str.strip()
df['Sleep Duration'] = df['Sleep Duration'].map({
    'Less than 5 hours': 4,
    '5-6 hours': 5.5,
    '7-8 hours': 7.5,
    'More than 8 hours': 9
})

df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Have you ever had suicidal thoughts ?'] = df['Have you ever had suicidal thoughts ?'].map({'Yes': 1, 'No': 0})
df['Family History of Mental Illness'] = df['Family History of Mental Illness'].map({'Yes': 1, 'No': 0})

df = pd.get_dummies(df, columns=['City', 'Dietary Habits', 'Degree'], drop_first=True, dtype=int)

X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(medians).astype(float)
X_test = X_test.fillna(medians).astype(float)

columns_to_scale = [
    'Age', 'Academic Pressure', 'CGPA', 'Study Satisfaction',
    'Sleep Duration', 'Work/Study Hours', 'Financial Stress'
]

scaler = StandardScaler()
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[1921  391]
 [ 501 2760]]
              precision    recall  f1-score   support

           0       0.79      0.83      0.81      2312
           1       0.88      0.85      0.86      3261

    accuracy                           0.84      5573
   macro avg       0.83      0.84      0.84      5573
weighted avg       0.84      0.84      0.84      5573

